In [ ]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()

    result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [ ]:
llm= EasyLLM(provider="openai_responses",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high","summary":"auto"},verbose_thinking=True)

In [36]:
from core.Message import UserMessage


result=llm.invoke_raw([{"role":"user","content":"你好！"}])


2026-04-17 02:03:22,396 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/responses "HTTP/1.1 200 OK"
2026-04-17 02:03:22,397 | INFO | ✅ openairesponses Provider 原始响应成功


In [ ]:
result = llm.invoke_raw([{"role": "user", "content": "你好！"}])

new_message = [{"role": "user", "content": "你好！"}]
new_message.extend(result.output)
new_message.append({"role": "user", "content": "你是谁？"})


In [56]:
assistant_text = llm.provider.get_response_content(result)

msg_plain = [
    {"role": "user", "content": "你好！"},
    {"role": "assistant", "content": assistant_text},
    {"role": "user", "content": "你是谁？"},
]
msg_plain

[{'role': 'user', 'content': '你好！'},
 {'role': 'assistant',
  'content': '\n\n你好！很高兴见到你！😊\n\n有什么我可以帮你的吗？无论是回答问题、提供建议，还是随便聊聊，我都在这里～'},
 {'role': 'user', 'content': '你是谁？'}]

In [57]:
llm.invoke_raw(messages=msg_plain)

2026-04-17 02:18:16,345 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/responses "HTTP/1.1 200 OK"
2026-04-17 02:18:16,347 | INFO | ✅ openairesponses Provider 原始响应成功


Response(id='resp_a801c8c6af1bb034', created_at=1776363494.0, error=None, incomplete_details=None, instructions=None, metadata=None, model='qwen3.5-9b', object='response', output=[ResponseReasoningItem(id='rs_918a96aa6401598e', summary=[], type='reasoning', content=[Content(text='Thinking Process:\n\n1.  **Identify the user\'s question**: The user is asking "你是谁？" (Who are you?).\n', type='reasoning_text')], encrypted_content=None, status=None), ResponseOutputMessage(id='msg_8aeca2535d8df9aa', content=[ResponseOutputText(annotations=[], text='\n\n你好！我是 Qwen3.5，是阿里巴巴集团最新推出的通义千问大语言模型。我具备强大的语言理解、逻辑推理、代码编写及分析等能力，可以协助你完成写作、编程、数据分析、多语言翻译、内容创作等任务。如果你有任何问题或需要帮助，随时告诉我哦！', type='output_text', logprobs=None)], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=None, conversation=None, max_output_tokens=262093, max_tool_calls=None, previous_response_id=None, prompt=No

In [ ]:
test_invoke_without_tool(agent)

In [ ]:
result=agent.invoke("你好，请介绍一下你自己")
print(result)

In [ ]:
agent.get_history()

In [ ]:
await test_ainvoke_without_tool(agent)

In [ ]:
test_stream_without_tool(agent)

In [ ]:
await test_astream_without_tool(agent)

In [ ]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

In [ ]:
test_invoke_with_tool(agent)

In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [ ]:
await test_astream_with_tool(agent)

In [ ]:
agent.history